# Q3: Early Behavioural Signals and Later Intent / Switching Outcomes

Do the actions a customer takes in the months after first contact predict whether they'll show intent to switch or actually switch their mortgage?

**Sections:**
1. [Setup](#1-setup)
2. [Descriptive Analysis (EDA)](#2-descriptive-analysis)
3. [Data Quality](#3-data-quality)
4. [Analytical Unit — Mortgage-Level Base Table](#4-analytical-unit)
5. [Funnel Analysis](#5-funnel-analysis)
6. [Signal Strength](#6-signal-strength)
7. [Engagement Timing](#7-engagement-timing)
8. [Limitations and Further Evidence](#8-limitations)

---
## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

In [ ]:
mortgages  = pd.read_csv('../seeds/mortgages.csv')
applicants = pd.read_csv('../seeds/applicants.csv')
events     = pd.read_csv('../seeds/events.csv')

# parse dates upfront
mortgages['mortgage_expiry_date']  = pd.to_datetime(mortgages['mortgage_expiry_date'])
mortgages['mortgage_closed_date']  = pd.to_datetime(mortgages['mortgage_closed_date'])
events['event_created_date']       = pd.to_datetime(events['event_created_date'])

print(f'mortgages : {mortgages.shape}')
print(f'applicants: {applicants.shape}')
print(f'events    : {events.shape}')

In [ ]:
# funnel stage order — used across multiple charts
FUNNEL_STAGES = [
    'email_sent',
    'email_clicked',
    'logged_in',
    'subtopic_read',
    'start_review',
    'intent'
]

FUNNEL_LABELS = {
    'email_sent'    : 'Email sent',
    'email_clicked' : 'Clicked email',
    'logged_in'     : 'Logged in',
    'subtopic_read' : 'Read content',
    'start_review'  : 'Started review',
    'intent'        : 'Showed intent'
}

STATE_COLORS = {
    'open'     : '#5B8DB8',
    'switched' : '#E07B54',
    'redeemed' : '#68A96A'
}

---
## 2. Descriptive Analysis

### 2.1 Mortgage portfolio

In [ ]:
print(mortgages.dtypes)
display(mortgages.head())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# mortgage state
state_counts = mortgages['mortgage_state'].value_counts()
axes[0].bar(state_counts.index, state_counts.values,
            color=[STATE_COLORS.get(s, '#999') for s in state_counts.index])
axes[0].set_title('Mortgage state')
axes[0].set_ylabel('Count')
for i, v in enumerate(state_counts.values):
    axes[0].text(i, v + 20, str(v), ha='center', fontsize=9)

# mortgage type
type_counts = mortgages['mortgage_type'].value_counts()
axes[1].barh(type_counts.index, type_counts.values, color='#5B8DB8')
axes[1].set_title('Mortgage type')
axes[1].set_xlabel('Count')

# lender
lender_counts = mortgages['firm_alias'].value_counts()
axes[2].bar(lender_counts.index, lender_counts.values, color='#7E9E7E')
axes[2].set_title('Mortgages by lender')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.suptitle('Portfolio overview', y=1.02, fontsize=13, fontweight='bold')
plt.show()

In [ ]:
# expiry date distribution — shows when the portfolio comes up for renewal
mortgages['expiry_year'] = mortgages['mortgage_expiry_date'].dt.year
expiry_dist = mortgages.groupby(['expiry_year', 'mortgage_state']).size().unstack(fill_value=0)

expiry_dist.plot(kind='bar', stacked=True,
                 color=[STATE_COLORS.get(c, '#999') for c in expiry_dist.columns],
                 figsize=(9, 4))
plt.title('Mortgage expiry year by state')
plt.xlabel('Expiry year')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.legend(title='State')
plt.tight_layout()
plt.show()

### 2.2 Event volume and timeline

In [ ]:
print(events.dtypes)
display(events.head())

In [ ]:
# event type volume
event_counts = events['event_alias'].value_counts().reindex(
    [e for e in FUNNEL_STAGES if e in events['event_alias'].unique()] +
    [e for e in events['event_alias'].unique() if e not in FUNNEL_STAGES]
).dropna()

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(event_counts.index[::-1], event_counts.values[::-1], color='#5B8DB8')
ax.set_xlabel('Event count')
ax.set_title('Event volume by type')
for i, v in enumerate(event_counts.values[::-1]):
    ax.text(v + 100, i, f'{v:,}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# monthly event volume per type — shows when engagement started and any trends
events['event_month'] = events['event_created_date'].dt.to_period('M')
monthly = events.groupby(['event_month', 'event_alias']).size().unstack(fill_value=0)
monthly.index = monthly.index.to_timestamp()

fig, ax = plt.subplots(figsize=(12, 4))
for col in FUNNEL_STAGES:
    if col in monthly.columns:
        ax.plot(monthly.index, monthly[col], marker='o', markersize=3, label=FUNNEL_LABELS[col])
ax.set_title('Monthly event volume by type')
ax.set_xlabel('')
ax.set_ylabel('Events')
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

### 2.3 Applicants bridge — many-to-many sizing

In [ ]:
print(applicants.dtypes)
display(applicants.head())

mortgages_per_consumer = applicants.groupby('applicant_consumer_id')['applicant_mortgage_id'].nunique()
consumers_per_mortgage = applicants.groupby('applicant_mortgage_id')['applicant_consumer_id'].nunique()

print('\nMortgages per consumer:')
print(mortgages_per_consumer.value_counts().sort_index().rename('consumers'))

print('\nConsumers per mortgage:')
print(consumers_per_mortgage.value_counts().sort_index().rename('mortgages'))

### 2.4 Missing values profile

In [ ]:
def null_profile(df, label):
    pct = (df.isnull().sum() / len(df) * 100).rename('null_%')
    counts = df.isnull().sum().rename('null_count')
    return pd.concat([counts, pct], axis=1).assign(table=label)

null_summary = pd.concat([
    null_profile(mortgages, 'mortgages'),
    null_profile(applicants, 'applicants'),
    null_profile(events, 'events')
])

display(null_summary[null_summary['null_count'] > 0].sort_values('null_%', ascending=False))

In [ ]:
# null event_mortgage_id broken down by event type — intent is expected to be near 100% null
null_by_type = (
    events
    .assign(mortgage_id_null=events['event_mortgage_id'].isna())
    .groupby('event_alias')['mortgage_id_null']
    .agg(['sum', 'count'])
    .assign(pct_null=lambda df: df['sum'] / df['count'] * 100)
    .rename(columns={'sum': 'null_count', 'count': 'total'})
    .sort_values('pct_null', ascending=False)
)
display(null_by_type)

---
## 3. Data Quality

In [ ]:
# DQ flag 1: mortgages recorded as 'open' but with a closed_date
dq_open_with_close = mortgages[
    (mortgages['mortgage_state'] == 'open') & mortgages['mortgage_closed_date'].notna()
]
print(f'Open mortgages with a closed_date: {len(dq_open_with_close)} '
      f'({len(dq_open_with_close)/len(mortgages)*100:.1f}%)')
print('Resolution: treat mortgage_state as authoritative; closed_date will be ignored for these rows.')
display(dq_open_with_close.head())

In [ ]:
# DQ flag 2: engagement events with no mortgage_id
engagement_types = ['email_clicked', 'logged_in', 'subtopic_read', 'start_review', 'intent']
dq_no_mortgage = events[
    events['event_alias'].isin(engagement_types) & events['event_mortgage_id'].isna()
]
print(f'Engagement events with null mortgage_id: {len(dq_no_mortgage):,} '
      f'({len(dq_no_mortgage)/len(events)*100:.1f}% of all events)')
print('\nBreakdown by type (intent events are structurally consumer-level, not a defect):')
display(dq_no_mortgage['event_alias'].value_counts())

In [ ]:
# DQ flag 3: mortgages with no applicant record (can never be attributed a signal)
mortgages_no_consumer = mortgages[
    ~mortgages['mortgage_id'].isin(applicants['applicant_mortgage_id'])
]
print(f'Mortgages with no consumer in applicants: {len(mortgages_no_consumer)} '
      f'({len(mortgages_no_consumer)/len(mortgages)*100:.1f}%)')

In [ ]:
# DQ flag 4: date sanity — any closed_date before expiry_date?
closed_before_expiry = mortgages[
    mortgages['mortgage_closed_date'].notna() &
    (mortgages['mortgage_closed_date'] < mortgages['mortgage_expiry_date'])
]
print(f'Mortgages closed before expiry: {len(closed_before_expiry)} '
      f'— these are expected (switching/redemption happens mid-term)')

---
## 4. Analytical Unit

One row per mortgage. All joins are **left joins** starting from `mortgages` so nothing is silently dropped. Nulls are preserved and categorised.

**Why left join instead of inner join?**  
An inner join (only mortgages that appear in applicants *and* have events) conditions the analysis on having been engaged. That inflates engagement rates because the denominator excludes mortgages that were never reached. Left joining and flagging nulls keeps the denominator honest.

In [ ]:
# one consumer per mortgage — use first applicant record if a mortgage has multiple
primary_consumer = (
    applicants
    .groupby('applicant_mortgage_id')['applicant_consumer_id']
    .first()
    .reset_index()
    .rename(columns={
        'applicant_mortgage_id': 'mortgage_id',
        'applicant_consumer_id': 'consumer_id'
    })
)

base = (
    mortgages
    .rename(columns={
        'mortgage_id'           : 'mortgage_id',
        'firm_alias'            : 'lender',
        'mortgage_type'         : 'mortgage_type',
        'mortgage_state'        : 'mortgage_state',
        'mortgage_expiry_date'  : 'expiry_date',
        'mortgage_closed_date'  : 'closed_date'
    })
    .merge(primary_consumer, on='mortgage_id', how='left')
)

print(f'Base after consumer join: {len(base)} rows')

In [ ]:
# first email per mortgage (email_sent events carry a mortgage_id directly)
first_email = (
    events[events['event_alias'] == 'email_sent']
    .groupby('event_mortgage_id')['event_created_date']
    .min()
    .reset_index()
    .rename(columns={
        'event_mortgage_id'  : 'mortgage_id',
        'event_created_date' : 'first_email_date'
    })
)

base = base.merge(first_email, on='mortgage_id', how='left')

In [ ]:
# intent events do not carry a mortgage_id — resolve via consumer bridge
# if a consumer holds multiple mortgages, intent is attributed to all of them
intent_via_consumer = (
    events[events['event_alias'] == 'intent']
    .merge(
        applicants.rename(columns={
            'applicant_consumer_id': 'event_consumer_id',
            'applicant_mortgage_id': 'mortgage_id'
        }),
        on='event_consumer_id',
        how='inner'
    )
    .groupby('mortgage_id')['event_created_date']
    .min()
    .reset_index()
    .rename(columns={'event_created_date': 'first_intent_date'})
)

base = base.merge(intent_via_consumer, on='mortgage_id', how='left')

# how many mortgages were touched by the multi-mortgage consumer attribution?
multi_mortgage_consumers = applicants.groupby('applicant_consumer_id')['applicant_mortgage_id'].nunique()
n_multi = (multi_mortgage_consumers > 1).sum()
print(f'Consumers with >1 mortgage: {n_multi} — intent events for these consumers '
      f'are attributed to all their mortgages')

In [ ]:
# engagement flags — events that carry mortgage_id directly
direct_events = ['email_clicked', 'logged_in', 'subtopic_read', 'start_review']

for alias in direct_events:
    reached = (
        events[events['event_alias'] == alias]
        .dropna(subset=['event_mortgage_id'])
        .groupby('event_mortgage_id').size()
        .reset_index(name='_cnt')
        .rename(columns={'event_mortgage_id': 'mortgage_id'})
        .assign(**{f'reached_{alias}': 1})
        [['mortgage_id', f'reached_{alias}']]
    )
    base = base.merge(reached, on='mortgage_id', how='left')
    base[f'reached_{alias}'] = base[f'reached_{alias}'].fillna(0).astype(int)

# derived binary flags
base['had_intent']           = base['first_intent_date'].notna().astype(int)
base['is_switched']          = (base['mortgage_state'] == 'switched').astype(int)
base['in_cohort']            = base['first_email_date'].notna().astype(int)

# funnel stage: highest stage reached (0 = no engagement)
FUNNEL_FLAG_COLS = ['reached_email_clicked', 'reached_logged_in',
                    'reached_subtopic_read', 'reached_start_review', 'had_intent']
base['funnel_stage'] = base[FUNNEL_FLAG_COLS].sum(axis=1).astype(int)

# add email_sent as stage 0 baseline
base['reached_email_sent'] = base['in_cohort']

# days to key milestones (only valid where cohorted)
base['days_to_intent'] = (base['first_intent_date'] - base['first_email_date']).dt.days
base['days_to_close']  = (base['closed_date']       - base['first_email_date']).dt.days

print(f'Base table shape: {base.shape}')
display(base.head())

In [ ]:
# analytical universe waterfall — shows exactly how many mortgages are in scope at each stage
waterfall = pd.DataFrame([
    {'Universe'                                : 'All mortgages',
     'Count'                                   : len(base)},
    {'Universe'                                : 'Has a consumer (in applicants)',
     'Count'                                   : base['consumer_id'].notna().sum()},
    {'Universe'                                : 'Has at least one event (via consumer)',
     'Count'                                   : base[
         base['consumer_id'].isin(events['event_consumer_id'])
     ]['mortgage_id'].nunique()},
    {'Universe'                                : 'Has first email_sent (cohort)',
     'Count'                                   : base['in_cohort'].sum()},
    {'Universe'                                : 'Showed intent',
     'Count'                                   : base['had_intent'].sum()},
    {'Universe'                                : 'Switched',
     'Count'                                   : base['is_switched'].sum()},
])
waterfall['% of total'] = (waterfall['Count'] / len(base) * 100).round(1)
display(waterfall)

---
## 5. Funnel Analysis

The cohort is mortgages with a `first_email_date` — i.e. the lender sent at least one email. All funnel rates use this cohort as the denominator.

In [ ]:
cohort = base[base['in_cohort'] == 1].copy()
print(f'Cohort size (mortgages with email_sent): {len(cohort)}')

In [ ]:
# Chart 1: Funnel volume + stage-to-stage conversion rates

flag_cols = [
    'reached_email_sent',
    'reached_email_clicked',
    'reached_logged_in',
    'reached_subtopic_read',
    'reached_start_review',
    'had_intent'
]

funnel_counts = {s: cohort[f].sum() for s, f in zip(FUNNEL_STAGES, flag_cols)}
funnel_df = pd.DataFrame(list(funnel_counts.items()), columns=['stage', 'count'])
funnel_df['label'] = funnel_df['stage'].map(FUNNEL_LABELS)
funnel_df['pct_of_total'] = funnel_df['count'] / len(cohort) * 100
funnel_df['stage_to_stage'] = funnel_df['count'].pct_change() * 100

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(
    funnel_df['label'][::-1],
    funnel_df['count'][::-1],
    color='#5B8DB8'
)

for i, (_, row) in enumerate(funnel_df[::-1].iterrows()):
    ax.text(row['count'] + 30, i, f"{row['count']:,}  ({row['pct_of_total']:.0f}%)",
            va='center', fontsize=9)

# annotate drop-offs between stages
for i in range(1, len(funnel_df)):
    prev = funnel_df.iloc[i - 1]['count']
    curr = funnel_df.iloc[i]['count']
    drop = (1 - curr / prev) * 100 if prev > 0 else 0
    # position in reversed chart: stage i is at position len-1-i from the top
    y_pos = len(funnel_df) - 1 - i + 0.5
    ax.text(max(funnel_df['count']) * 0.5, y_pos,
            f'▼ {drop:.0f}% drop',
            va='center', ha='center', fontsize=8, color='#cc4400',
            style='italic')

ax.set_xlabel('Mortgages')
ax.set_title('Engagement funnel — cohort mortgages only', fontweight='bold')
ax.set_xlim(0, max(funnel_df['count']) * 1.35)
plt.tight_layout()
plt.show()

In [ ]:
# Chart 2: Funnel by lender — do all lenders have the same shape?
lenders = cohort['lender'].unique()
fig, axes = plt.subplots(1, len(lenders), figsize=(14, 4), sharey=True)

for ax, lender in zip(axes, lenders):
    sub = cohort[cohort['lender'] == lender]
    counts = [sub[f].sum() for f in flag_cols]
    pcts   = [c / len(sub) * 100 for c in counts]
    ax.barh([FUNNEL_LABELS[s] for s in FUNNEL_STAGES][::-1],
            pcts[::-1], color='#5B8DB8')
    ax.set_title(lender, fontsize=10)
    ax.set_xlabel('% of cohort')

fig.suptitle('Funnel conversion by lender (%)', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6. Signal Strength

Does reaching deeper into the funnel actually predict switching?

In [ ]:
# Chart 3: Funnel stage reached × mortgage outcome (stacked 100% bar)

stage_label_map = {
    0: 'No engagement',
    1: 'Clicked email',
    2: 'Logged in',
    3: 'Read content',
    4: 'Started review',
    5: 'Showed intent'
}

# include all mortgages (not just cohort) so 'no engagement' bar is meaningful
stage_outcome = (
    base
    .assign(stage_label=base['funnel_stage'].map(stage_label_map))
    .groupby(['stage_label', 'mortgage_state'])
    .size()
    .unstack(fill_value=0)
)
stage_pct = stage_outcome.div(stage_outcome.sum(axis=1), axis=0) * 100

# order stages logically
stage_order = [stage_label_map[i] for i in range(6) if stage_label_map[i] in stage_pct.index]
stage_pct = stage_pct.reindex(stage_order)

ax = stage_pct.plot(
    kind='bar', stacked=True,
    color=[STATE_COLORS.get(c, '#999') for c in stage_pct.columns],
    figsize=(10, 5),
    width=0.6
)
ax.set_xlabel('Highest funnel stage reached')
ax.set_ylabel('% of mortgages')
ax.set_title('Mortgage outcome by funnel stage reached', fontweight='bold')
ax.set_xticklabels(stage_order, rotation=25, ha='right')
ax.legend(title='State', bbox_to_anchor=(1, 1))

# annotate switch % on each bar
for i, stage in enumerate(stage_order):
    if 'switched' in stage_pct.columns:
        switch_pct = stage_pct.loc[stage, 'switched']
        ax.text(i, 101, f'{switch_pct:.0f}%', ha='center', fontsize=8, color='#E07B54')

plt.tight_layout()
plt.show()

In [ ]:
# Chart 4: Intent flag vs switch rate — the headline signal

intent_switch = (
    base
    .groupby('had_intent')['is_switched']
    .agg(['sum', 'count'])
    .assign(switch_rate=lambda df: df['sum'] / df['count'] * 100)
    .reset_index()
)
intent_switch['label'] = intent_switch['had_intent'].map({0: 'No intent signal', 1: 'Showed intent'})

# odds ratio
r_intent     = intent_switch.loc[intent_switch['had_intent'] == 1, 'switch_rate'].values[0] / 100
r_no_intent  = intent_switch.loc[intent_switch['had_intent'] == 0, 'switch_rate'].values[0] / 100
odds_ratio   = (r_intent / (1 - r_intent)) / (r_no_intent / (1 - r_no_intent))

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(
    intent_switch['label'],
    intent_switch['switch_rate'],
    color=['#5B8DB8', '#E07B54'],
    width=0.4
)
for bar, rate in zip(bars, intent_switch['switch_rate']):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.5,
            f'{rate:.1f}%', ha='center', fontsize=10, fontweight='bold')

ax.set_ylabel('Switch rate (%)')
ax.set_title('Switch rate: intent signal vs no intent signal', fontweight='bold')
ax.set_ylim(0, max(intent_switch['switch_rate']) * 1.3)
ax.text(0.5, -0.18,
        f'Odds ratio: {odds_ratio:.1f}x  — '
        f'mortgages with intent are {odds_ratio:.1f}x more likely to switch',
        transform=ax.transAxes, ha='center', fontsize=9, color='#555')
plt.tight_layout()
plt.show()

In [ ]:
# Chart 5: Days-to-intent distribution

days_data = base[
    base['had_intent'] == 1
]['days_to_intent'].dropna()

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(days_data, bins=40, color='#5B8DB8', edgecolor='white')

med   = days_data.median()
p90   = days_data.quantile(0.90)
ax.axvline(med, color='#E07B54', linestyle='--', label=f'Median: {med:.0f}d')
ax.axvline(p90, color='#cc4400', linestyle=':',  label=f'P90: {p90:.0f}d')

ax.set_xlabel('Days from first email to intent event')
ax.set_ylabel('Mortgages')
ax.set_title('When does intent emerge? (days from first email)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Median days to intent: {med:.0f}')
print(f'90% of intent occurs within {p90:.0f} days')

In [ ]:
# Chart 6: Cohort consistency heatmap — are signals stable across cohort months?

cohort['cohort_month'] = cohort['first_email_date'].dt.to_period('M')

cohort_metrics = (
    cohort
    .groupby('cohort_month')
    .agg(
        n            =('mortgage_id', 'count'),
        intent_rate  =('had_intent',  'mean'),
        switch_rate  =('is_switched', 'mean'),
        click_rate   =('reached_email_clicked', 'mean')
    )
    .query('n >= 10')  # exclude tiny cohorts
    .mul({'n': 1, 'intent_rate': 100, 'switch_rate': 100, 'click_rate': 100})
    .drop(columns='n')
)
cohort_metrics.index = cohort_metrics.index.to_timestamp().strftime('%b %Y')

fig, ax = plt.subplots(figsize=(8, len(cohort_metrics) * 0.5 + 1.5))
sns.heatmap(
    cohort_metrics,
    annot=True, fmt='.0f', cmap='YlOrRd',
    linewidths=0.4, ax=ax,
    cbar_kws={'label': '%'}
)
ax.set_title('Key rates by cohort month (% of cohort)', fontweight='bold')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

---
## 7. Engagement Timing

Does *when* a customer engages matter? Month 1 engagement vs months 2 and 3.

In [ ]:
# assign each event to month 1, 2, or 3 relative to the mortgage's first email
email_cohort_map = cohort.set_index('mortgage_id')['first_email_date'].to_dict()

engagement_events = events[
    events['event_alias'].isin(['email_clicked', 'logged_in', 'subtopic_read', 'start_review'])
    & events['event_mortgage_id'].notna()
].copy()

engagement_events['first_email'] = engagement_events['event_mortgage_id'].map(email_cohort_map)
engagement_events = engagement_events.dropna(subset=['first_email'])
engagement_events['months_since_email'] = (
    (engagement_events['event_created_date'].dt.to_period('M') -
     engagement_events['first_email'].dt.to_period('M')).apply(lambda x: x.n)
) + 1

# flag mortgages that engaged in each month window
for m in [1, 2, 3]:
    engaged_m = (
        engagement_events[engagement_events['months_since_email'] == m]
        ['event_mortgage_id'].unique()
    )
    cohort[f'engaged_m{m}'] = cohort['mortgage_id'].isin(engaged_m).astype(int)

print(cohort[['engaged_m1', 'engaged_m2', 'engaged_m3']].sum())

In [ ]:
# Chart 7: Month 1/2/3 engagement vs intent rate

timing_results = []
for m in [1, 2, 3]:
    col = f'engaged_m{m}'
    for engaged in [0, 1]:
        sub = cohort[cohort[col] == engaged]
        timing_results.append({
            'month'      : f'Month {m}',
            'engaged'    : 'Engaged' if engaged else 'Not engaged',
            'intent_rate': sub['had_intent'].mean() * 100,
            'switch_rate': sub['is_switched'].mean() * 100,
            'n'          : len(sub)
        })

timing_df = pd.DataFrame(timing_results)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, metric, title in zip(
    axes,
    ['intent_rate', 'switch_rate'],
    ['Intent rate', 'Switch rate']
):
    pivot = timing_df.pivot(index='month', columns='engaged', values=metric)
    pivot.plot(kind='bar', ax=ax, color=['#aac4de', '#E07B54'], width=0.6)
    ax.set_title(f'{title} by engagement timing', fontweight='bold')
    ax.set_ylabel('%')
    ax.set_xlabel('')
    ax.set_xticklabels(pivot.index, rotation=0)
    ax.legend(title='Engagement in month')

plt.tight_layout()
plt.show()

---
## 8. Limitations and Further Evidence

### What the data shows

There is a clear association between reaching deeper funnel stages — particularly the `intent` event — and higher switch rates. The odds ratio computed in Section 6 quantifies the strength of this signal. The cohort heatmap shows whether this pattern holds consistently across time or is driven by a single cohort.

### What the data cannot tell us

**1. Selection into the cohort**  
Only mortgages that received an email enter the funnel. We don't know why some mortgages were never emailed — if it correlates with lender configuration or product type, the funnel rates are confounded before we even start.

**2. Causation vs correlation**  
The `intent` event likely reflects a decision the customer had already made, triggered by external factors (rate environment, house move, life event). Engagement may be tracking that decision, not driving it. A customer who intends to switch will click the email; the click doesn't cause the intent.

**3. Short observation window**  
Events start May 2024. Mortgages cohorted in late 2024 have not had enough time to generate a `switched` or `redeemed` outcome. Their switch rates will be understated, which can make recent cohorts look weaker than older ones.

**4. Missing economic context**  
There is no data on the current mortgage rate vs available market rates, loan-to-value, or consumer income. The incentive to switch is almost entirely determined by the rate differential — if we don't control for that, engagement effects are impossible to isolate.

### What would change the conclusion

| Evidence | Why it matters |
|---|---|
| A/B test (randomise email outreach) | Only way to establish whether engagement causes switching, not just correlates |
| Longer time horizon (2+ year outcome window) | Let recent cohorts mature so switch rates are observable |
| Rate delta per mortgage | Controls for the main economic driver of switching |
| Consumer demographics | Controls for selection — who gets emailed and why |
| Exit survey or CRM notes on why switchers switched | Disambiguates engagement-driven vs external-driver switching |